# 第 1 周练习解答 —— 技术问题解释器

## 练习目标（理念）

用 **OpenRouter**（云端 `gpt-4o-mini`）和本地 **Ollama**（`llama3.2:1b`）各跑一遍「解释一段 Python」：

- **输入**：一段看不懂的代码 / 技术问题
- **输出**：清晰的技术解释
- **对比**：云端模型流式输出 vs 本地模型整段返回

这和 Day 1 / Day 2 的 Chat Completions、`messages`、`stream=True`、Ollama OpenAI 兼容端点一脉相承。

## 怎么跑

1. 准备环境变量 `OPENROUTER_API_KEY`（本笔记本用 `os.getenv`，未写 `load_dotenv`，可在 shell 里 export，或自行加 dotenv）
2. 本机启动 Ollama，并确保已拉取 `llama3.2:1b`
3. 从上到下运行单元格；可改 `question` 再对比两种后端


In [ ]:
# ========== 导入：后面要用的工具箱 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENROUTER_API_KEY
import os
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 可指向 OpenRouter 或本地 Ollama
from openai import OpenAI


In [ ]:
# ========== 常量：模型名字集中写在一处 ==========

# OpenRouter 侧使用的模型 id（字符串必须和平台上的模型名一致）
MODEL_GPT = "gpt-4o-mini"
# 本地 Ollama 模型名：带 :1b 标签，需事先 ollama pull llama3.2:1b
MODEL_LLAMA = "llama3.2:1b"


In [ ]:
# ========== 两个客户端：OpenRouter（云）+ Ollama（本地，OpenAI 兼容）==========

# 开放路由器设置：从环境变量读取 API Key（不要把密钥写进笔记本）
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# 没有密钥就立刻失败，避免后面请求时才报含糊错误
if not OPENROUTER_API_KEY:
    # 错误文案保留英文原样：可能被脚本/测试依赖
    raise ValueError("OPENROUTER_API_KEY not found.")

# 创建指向 OpenRouter 的 OpenAI 兼容客户端：base_url 改成 OpenRouter 的 v1 根路径
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Ollama 设置（OpenAI 兼容端点）：本地 11434 端口的 /v1
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Ollama 不需要真正的 API 密钥，但 OpenAI SDK 要求传一个非空 api_key
OLLAMA_API_KEY = "ollama"

# 第二个客户端：同一套 chat.completions API，打到本机 Ollama
ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY
)


In [ ]:
# ========== 提问：要解释的技术问题（user 消息内容）==========

# question 字符串会原样发给模型；改这里即可换题（prompt 正文保持英文原样）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 路径 A：OpenRouter + GPT，流式（streaming）边收边打印 ==========

# 打印分隔标题，方便在笔记本输出里区分两家后端
print("=== GPT-4o-mini (OpenRouter) ===\n")

# chat.completions.create：发起 Chat Completions；stream=True 返回可迭代的增量块
stream = openrouter.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        # system：定角色/风格；content 是发给模型的指令，保留英文不译
        {"role": "system", "content": "You are a technical Python expert."},
        # user：真正的问题（上面定义的 question）
        {"role": "user", "content": question}
    ],
    stream=True
)

# 遍历流式 chunk：每个 chunk 可能带一小段 delta.content
for chunk in stream:
    # 取出本块增量文本；结束或空块时可能为 None
    content = chunk.choices[0].delta.content
    if content:
        # end="" 不换行；flush=True 立刻刷到屏幕，形成「打字机」效果
        print(content, end="", flush=True)

# 流结束后补一个空行，和后面 Llama 输出隔开
print("\n")


In [ ]:
# ========== 路径 B：本地 Ollama + Llama，整段（非流式）一次返回 ==========

print("=== Llama 3.2 (Ollama Local) ===\n")

# 同样走 chat.completions.create，但客户端是 ollama；默认 stream=False，等整段完成
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        # system / user 与上面 GPT 路径保持同一套消息，便于对比回答风格
        {"role": "system", "content": "You are a technical Python expert."},
        {"role": "user", "content": question}
    ]
)

# 非流式：完整答案在 message.content 里，一次性打印
print(response.choices[0].message.content)
